# YOLOv26n + DAFE on NEU-DET — Steel Defect Detection

## Goal
Train YOLOv26n + our DAFE (Defect-Aware Feature Enhancement) module on NEU-DET.
Compare against the YOLOv26n-only baseline (81.0% mAP@0.5).

## Architecture: YOLOv26n + DAFE
| Component | Description |
|-----------|------------|
| Base | YOLOv26n (NMS-free, DFL-free, C2PSA, C3k2) |
| DAFE@P2 | After C3k2 at P2 (128ch) — edge + texture enhancement for fine defects |
| DAFE@P3 | After C3k2 at P3 (256ch) — edge + texture enhancement for medium defects |
| Extra Params | ~100K (DAFE is lightweight) |

## DAFE Module (Novel Contribution)
- **Edge branch**: Sobel-initialized convolutions for linear defects (scratches, crazing)
- **Texture branch**: Local variance for surface anomalies (pitting, scale, inclusions)
- **Channel attention**: Fuses both branches with learned attention
- **Learnable residual**: Alpha parameter controls enhancement strength

## Training Recipe
Same optimized recipe as both baselines for fair comparison.

In [22]:
# Cell 1: Upgrade Ultralytics, Register DAFE & Environment Setup
import subprocess, sys

# Upgrade ultralytics to get YOLO26 support
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'ultralytics>=8.4.0'])
print('Ultralytics upgraded. Restart kernel if prompted.')

# Add project root to path
from pathlib import Path
ROOT = Path(r'D:/DigiSteel-Yolo/DigiSteel-YOLO')
sys.path.insert(0, str(ROOT))

# Register DAFE custom module BEFORE loading any YAML
from digisteel.engine.trainer import register_custom_modules
register_custom_modules()
print('Custom modules registered: DAFE, CoordAttention, GhostConv, WFCA, EMA')

import torch
import ultralytics
import json
import os
import random
import numpy as np

# Seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'Python version:    {sys.version.split()[0]}')
print(f'PyTorch version:   {torch.__version__}')
print(f'Ultralytics ver:   {ultralytics.__version__}')
print(f'CUDA available:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:               {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory:        {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — training will be extremely slow!')

Ultralytics upgraded. Restart kernel if prompted.
Custom modules registered: DAFE, CoordAttention, GhostConv, WFCA, EMA
Python version:    3.11.15
PyTorch version:   2.6.0+cu124
Ultralytics ver:   8.4.95
CUDA available:    True
GPU:               NVIDIA RTX 2000 Ada Generation
GPU Memory:        17.2 GB


In [23]:
# Cell 2: Path Setup & Verification
from pathlib import Path
import yaml

# Paths
ROOT = Path(r'D:/DigiSteel-Yolo/DigiSteel-YOLO')
DATA_YAML = ROOT / 'configs' / 'data' / 'neu_det.yaml'
MODEL_YAML = ROOT / 'configs' / 'models' / 'yolov26n_dafe.yaml'
RUNS_DIR = ROOT / 'runs' / 'detect'
RUN_NAME = 'yolov26n_dafe_neudet'
BEST_PT = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
RESULTS_CSV = RUNS_DIR / RUN_NAME / 'results.csv'
EVALS_DIR = ROOT / 'evals'
METRICS_JSON = EVALS_DIR / 'yolov26n_dafe_neudet_results.json'
TRAIN_STATUS_FILE = EVALS_DIR / 'yolov26n_dafe_neudet_train_status.json'

# Verify critical files
print('=' * 60)
print('PATH VERIFICATION')
print('=' * 60)

errors = []
if not DATA_YAML.exists():
    errors.append(f'Data YAML not found: {DATA_YAML}')
else:
    print(f'  Data YAML:   {DATA_YAML}')
    with open(DATA_YAML, 'r') as f:
        data_cfg = yaml.safe_load(f)
    base_path = Path(data_cfg.get('path', ''))
    if not base_path.is_absolute():
        base_path = DATA_YAML.parent / base_path
    for split in ['train', 'val', 'test']:
        split_rel = data_cfg.get(split, '')
        if split_rel:
            split_path = base_path / split_rel
            if split_path.exists():
                count = len(list(split_path.glob('*.jpg'))) + len(list(split_path.glob('*.png')))
                print(f'  {split}: {split_path} ({count} images)')
            else:
                errors.append(f'{split} path not found: {split_path}')

if not MODEL_YAML.exists():
    errors.append(f'Model YAML not found: {MODEL_YAML}')
else:
    print(f'  Model YAML:  {MODEL_YAML}')

EVALS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Load YOLOv26n-only baseline for comparison
BASELINE_JSON = EVALS_DIR / 'yolov26n_neudet_results.json'
if BASELINE_JSON.exists():
    import json as _json
    with open(BASELINE_JSON) as f:
        baseline = _json.load(f)
    print(f'\n  Baseline (YOLOv26n): {BASELINE_JSON}')
    print(f'  Baseline mAP@0.5:    {baseline["map50"]*100:.1f}%')
    print(f'  Baseline mAP@50:95:  {baseline["map50_95"]*100:.1f}%')
else:
    print(f'\n  WARNING: Baseline not found at {BASELINE_JSON}')
    print(f'  Run yolov26_neudet.ipynb first for comparison.')
    baseline = None

if errors:
    print('\nERRORS:')
    for e in errors:
        print(f'  {e}')
    raise FileNotFoundError('Critical paths missing.')

print('\nAll checks passed. Ready to train.')

PATH VERIFICATION
  Data YAML:   D:\DigiSteel-Yolo\DigiSteel-YOLO\configs\data\neu_det.yaml
  train: D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\images\train (1290 images)
  val: D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\images\val (344 images)
  test: D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\images\test (166 images)
  Model YAML:  D:\DigiSteel-Yolo\DigiSteel-YOLO\configs\models\yolov26n_dafe.yaml

  Baseline (YOLOv26n): D:\DigiSteel-Yolo\DigiSteel-YOLO\evals\yolov26n_neudet_results.json
  Baseline mAP@0.5:    81.0%
  Baseline mAP@50:95:  43.1%

All checks passed. Ready to train.


In [24]:
# Cell 3: Build YOLOv26n + DAFE Model with Proper Weight Transfer
from ultralytics import YOLO
from digisteel.engine.weight_transfer import transfer_yolov26n_weights_to_dafe
import torch

# Build base YOLOv26n with full pretrained weights
print('Loading pretrained YOLOv26n...')
base_model = YOLO('yolo26n.pt')

# Build DAFE model from custom YAML (DAFE uses eager init with actual channels)
print('Building YOLOv26n + DAFE from custom YAML...')
dafe_model = YOLO(str(MODEL_YAML))

# Transfer weights positionally (DAFE layers keep their Sobel init)
print('\nTransferring weights from base to DAFE model...')
stats = transfer_yolov26n_weights_to_dafe(base_model.model, dafe_model.model, verbose=True)

# Expected: Detect head (nc=80 -> nc=6) will have ~102 failed tensors — this is NORMAL
detect_fails = stats['failed']
if detect_fails > 0:
    print(f'\nNote: {detect_fails} Detect head tensors skipped (COCO nc=80 vs NEU-DET nc=6) — expected.')

# Print model info
print('\n' + '=' * 60)
print('YOLOv26n + DAFE MODEL INFO')
print('=' * 60)
total_params = sum(p.numel() for p in dafe_model.model.parameters())
trainable_params = sum(p.numel() for p in dafe_model.model.parameters() if p.requires_grad)
base_params = sum(p.numel() for p in base_model.model.parameters())
print(f'Architecture:  YOLOv26n + DAFE (P2 + P3)')
print(f'Pretrained:    yolo26n.pt (positional transfer)')
print(f'Total params:  {total_params:,}')
print(f'Base params:   {base_params:,}')
print(f'DAFE overhead: +{total_params - base_params:,} (+{(total_params - base_params)/base_params*100:.1f}%)')
print(f'Trainable:     {trainable_params:,}')
print(f'Layers:        {len(list(dafe_model.model.modules()))}')
print(f'NMS-free:      Yes (YOLOv26 end-to-end)')
print(f'DFL-free:      Yes (reg_max=1)')
print(f'Weight xfer:   {stats["transferred"]} tensors transferred, {stats["dafe_layers"]} DAFE layers')
print('=' * 60)

Loading pretrained YOLOv26n...
Building YOLOv26n + DAFE from custom YAML...

Transferring weights from base to DAFE model...
DAFE layers found at indices: [3, 6]
Transfer mapping: 24 layer pairs

  OK   [ 0-> 0] Conv         6 params transferred
  OK   [ 1-> 1] Conv         6 params transferred
  OK   [ 2-> 2] C3k2         24 params transferred
  OK   [ 3-> 4] Conv         6 params transferred
  OK   [ 4-> 5] C3k2         24 params transferred
  OK   [ 5-> 7] Conv         6 params transferred
  OK   [ 6-> 8] C3k2         54 params transferred
  OK   [ 7-> 9] Conv         6 params transferred
  OK   [ 8->10] C3k2         54 params transferred
  OK   [ 9->11] SPPF         12 params transferred
  OK   [10->12] C2PSA        42 params transferred
  SKIP [11->13]: no common keys
  SKIP [12->14]: no common keys
  OK   [13->15] C3k2         54 params transferred
  SKIP [14->16]: no common keys
  SKIP [15->17]: no common keys
  OK   [16->18] C3k2         54 params transferred
  OK   [17->19] Co

In [25]:
# Cell 4: DAFE Module Verification
# Verify DAFE builds with correct channels and Sobel init is preserved
import torch
from digisteel.modules import DAFE

print('DAFE Module Verification (eager init)')
print('=' * 60)

# Test with actual scaled channels (width=0.5)
for ch, name in [(64, 'P2 (64ch)'), (128, 'P3 (128ch)')]:
    dafe = DAFE(ch, reduction=8)
    x = torch.randn(1, ch, 80, 80)
    y = dafe(x)
    params = sum(p.numel() for p in dafe.parameters())
    print(f'  DAFE@{name}: input={x.shape} -> output={y.shape} | params={params:,}')
    assert y.shape == x.shape, f'Shape mismatch: {y.shape} != {x.shape}'

    # Verify Sobel init is present in edge branch
    sobel_x_weight = dafe.edge_branch.conv.weight[0, 0]
    expected_sobel = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
    sobel_match = torch.allclose(sobel_x_weight, expected_sobel)
    print(f'    Sobel-X init preserved: {sobel_match}')

# Test alpha (learnable residual scaling)
dafe = DAFE(64, reduction=8)
alpha = torch.sigmoid(dafe.alpha_raw).item()
print(f'\n  Initial alpha: {alpha:.4f} (sigmoid(-2.2) ~ 0.1)')
print(f'  This means DAFE starts with ~10% contribution, growing during training')

# Test that repeated forwards are consistent
x1 = torch.randn(1, 64, 80, 80)
y1 = dafe(x1)
y2 = dafe(x1)
assert torch.allclose(y1, y2), 'DAFE output changed between forwards!'
print(f'  Repeated forward consistency: OK')

# Verify optimizer will see all params
opt = torch.optim.AdamW(dafe.parameters())
n_groups = len(opt.param_groups[0]['params'])
print(f'  Optimizer param groups: {n_groups} tensors (all trainable)')
print('=' * 60)
print('All DAFE checks passed.')

DAFE Module Verification (eager init)
  DAFE@P2 (64ch): input=torch.Size([1, 64, 80, 80]) -> output=torch.Size([1, 64, 80, 80]) | params=24,833
    Sobel-X init preserved: True
  DAFE@P3 (128ch): input=torch.Size([1, 128, 80, 80]) -> output=torch.Size([1, 128, 80, 80]) | params=98,817
    Sobel-X init preserved: True

  Initial alpha: 0.0998 (sigmoid(-2.2) ~ 0.1)
  This means DAFE starts with ~10% contribution, growing during training
  Repeated forward consistency: OK
  Optimizer param groups: 12 tensors (all trainable)
All DAFE checks passed.


In [42]:
# Cell 5: Training — Same Optimized Recipe
from ultralytics import YOLO
from digisteel.engine.trainer import register_custom_modules
from digisteel.engine.weight_transfer import transfer_yolov26n_weights_to_dafe
import time
import json as _json

# Register custom modules and build model (cell-independent)
register_custom_modules()
base_model = YOLO('yolo26n.pt')
model = YOLO(str(MODEL_YAML))
transfer_yolov26n_weights_to_dafe(base_model.model, model.model, verbose=False)

# Verify DAFE params are visible to optimizer
dafe_params = sum(p.numel() for n, p in model.model.named_parameters() if 'edge_branch' in n or 'texture_branch' in n or 'fusion' in n or 'channel_att' in n)
print(f'DAFE trainable params: {dafe_params:,} (must be > 0!)')
assert dafe_params > 0, 'DAFE params not in model — optimizer will not train them!'
print('Model built with pretrained weights transferred. All DAFE params trainable.')

# Training hyperparameters — identical recipe for fair comparison
base_overrides = {
    'data': str(DATA_YAML),
    'task': 'detect',
    'epochs': 600,
    'patience': 150,
    'batch': 16,
    'imgsz': 800,
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 5,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'mosaic': 0.0,
    'mixup': 0.15,
    'copy_paste': 0.1,
    'copy_paste_mode': 'flip',
    'degrees': 10.0,
    'translate': 0.2,
    'scale': 0.6,
    'shear': 5.0,
    'perspective': 0.0,
    'flipud': 0.5,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'erasing': 0.4,
    'cos_lr': True,
    'deterministic': True,
    'close_mosaic': 10,
    'amp': True,
    'seed': 42,
    'workers': 8,
    'save': True,
    'save_period': 50,
    'plots': True,
    'project': str(RUNS_DIR),
    'name': RUN_NAME,
    'exist_ok': True,
    'verbose': True,
}

print('=' * 60)
print('TRAINING CONFIGURATION')
print('=' * 60)
print(f'Architecture:  YOLOv26n + DAFE (P2 + P3)')
print(f'Weights:       Pretrained yolo26n.pt (positional transfer)')
print(f'Dataset:       {DATA_YAML}')
print(f'Epochs:        {base_overrides["epochs"]}')
print(f'Patience:      {base_overrides["patience"]}')
print(f'Batch:         {base_overrides["batch"]}')
print(f'Image size:    {base_overrides["imgsz"]}')
print(f'Optimizer:     {base_overrides["optimizer"]}')
print(f'LR:            {base_overrides["lr0"]}')
print(f'Mosaic:        {base_overrides["mosaic"]} (disabled)')
print(f'Mixup:         {base_overrides["mixup"]}')
print(f'Copy-paste:    {base_overrides["copy_paste"]} ({base_overrides["copy_paste_mode"]})')
print(f'Output:        {RUNS_DIR / RUN_NAME}')
print('=' * 60)
print()

# Train with copy_paste, fallback to 0.0 if it fails
training_start = time.time()
copy_paste_used = base_overrides['copy_paste']

try:
    print(f'Starting training with copy_paste={copy_paste_used}...')
    results = model.train(**base_overrides)
except Exception as e:
    error_msg = str(e).lower()
    if 'copy_paste' in error_msg or 'segment' in error_msg or 'mask' in error_msg:
        print(f'\ncopy_paste={copy_paste_used} failed. Retrying without it...')
        print(f'Error: {e}')
        base_overrides.pop('copy_paste', None)
        copy_paste_used = 0.0
        register_custom_modules()
        base_model = YOLO('yolo26n.pt')
        model = YOLO(str(MODEL_YAML))
        transfer_yolov26n_weights_to_dafe(base_model.model, model.model, verbose=False)
        results = model.train(**base_overrides)
    else:
        raise

training_time = time.time() - training_start

# Save training status
_train_status = {
    'training_time_seconds': training_time,
    'copy_paste_used': copy_paste_used,
    'model': 'yolov26n_dafe.yaml',
    'pretrained': 'yolo26n.pt',
}
with open(TRAIN_STATUS_FILE, 'w') as _f:
    _json.dump(_train_status, _f, indent=2)

print(f'\n{"=" * 60}')
print('TRAINING COMPLETE')
print(f'{"=" * 60}')
print(f'Duration: {training_time / 3600:.1f} hours')
print(f'Copy-paste: {copy_paste_used}')
print(f'Weights: {BEST_PT}')
print('=' * 60)

DAFE trainable params: 123,648 (must be > 0!)
Model built with pretrained weights transferred. All DAFE params trainable.
TRAINING CONFIGURATION
Architecture:  YOLOv26n + DAFE (P2 + P3)
Weights:       Pretrained yolo26n.pt (positional transfer)
Dataset:       D:\DigiSteel-Yolo\DigiSteel-YOLO\configs\data\neu_det.yaml
Epochs:        600
Patience:      150
Batch:         16
Image size:    800
Optimizer:     AdamW
LR:            0.001
Mosaic:        0.0 (disabled)
Mixup:         0.15
Copy-paste:    0.1 (flip)
Output:        D:\DigiSteel-Yolo\DigiSteel-YOLO\runs\detect\yolov26n_dafe_neudet

Starting training with copy_paste=0.1...
Ultralytics 8.4.95  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, cop

KeyboardInterrupt: 

In [41]:
# Cell 5a: Clean up old broken run (if exists) before retraining
import shutil
from pathlib import Path

run_dir = RUNS_DIR / RUN_NAME
if run_dir.exists():
    print(f'Removing old run directory: {run_dir}')
    shutil.rmtree(run_dir)
    print('Old run deleted. Training will start fresh.')
else:
    print('No old run found. Ready for fresh training.')

No old run found. Ready for fresh training.


In [40]:
# Cell 6: Evaluate on Test Set
from ultralytics import YOLO
import json
from pathlib import Path

# Register custom modules (required for loading DAFE model from checkpoint)
from digisteel.engine.trainer import register_custom_modules
register_custom_modules()

# Load training status if available
import json as _json
if TRAIN_STATUS_FILE.exists():
    with open(TRAIN_STATUS_FILE) as _f:
        _status = _json.load(_f)
    training_time = _status.get('training_time_seconds', 0)
    copy_paste_used = _status.get('copy_paste_used', 0.1)
else:
    training_time = 0
    copy_paste_used = 0.1

# Find best weights
if not BEST_PT.exists():
    candidates = list((RUNS_DIR / RUN_NAME).rglob('best.pt'))
    if candidates:
        BEST_PT = candidates[0]
    else:
        raise FileNotFoundError(f'No best.pt found in {RUNS_DIR / RUN_NAME}')

print(f'Loading best weights: {BEST_PT}')
best_model = YOLO(str(BEST_PT))

# Evaluate on test set
print(f'\nRunning validation on test set...')
val_results = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=800,
    batch=16,
    device=0,
    plots=True,
    verbose=True,
)

# Extract metrics
map50 = float(val_results.box.map50)
map50_95 = float(val_results.box.map)
precision = float(val_results.box.mp)
recall = float(val_results.box.mr)

# Per-class AP@0.5
class_names = getattr(val_results, 'names', None) or best_model.names
per_class_ap50 = {}
for cls_id, cls_name in class_names.items():
    if cls_id < len(val_results.box.ap50):
        per_class_ap50[cls_name] = float(val_results.box.ap50[cls_id])

print(f'\n{"=" * 60}')
print('TEST SET RESULTS — YOLOv26n + DAFE')
print('=' * 60)
print(f'mAP@0.5:      {map50:.4f} ({map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {map50_95:.4f} ({map50_95*100:.1f}%)')
print(f'Precision:    {precision:.4f}')
print(f'Recall:       {recall:.4f}')
print(f'\nPer-class AP@0.5:')
print('-' * 40)
for cls_name, ap in per_class_ap50.items():
    print(f'  {cls_name:<20} {ap:.4f} ({ap*100:.1f}%)')
print('=' * 60)

FileNotFoundError: No best.pt found in D:\DigiSteel-Yolo\DigiSteel-YOLO\runs\detect\yolov26n_dafe_neudet

In [35]:
# Cell 7: DAFE vs YOLOv26n-Only Comparison
import json

# Ensure custom modules are registered (needed if kernel restarted)
from digisteel.engine.trainer import register_custom_modules
register_custom_modules()

# Save DAFE metrics (map50, map50_95, etc. come from Cell 6)
dafe_metrics = {
    'experiment': 'yolov26n_dafe_neudet',
    'model': 'yolov26n_dafe.yaml',
    'map50': map50,
    'map50_95': map50_95,
    'precision': precision,
    'recall': recall,
    'per_class_ap50': per_class_ap50,
    'training_time_hours': round(training_time / 3600, 2),
    'copy_paste_used': copy_paste_used,
    'hyperparameters': {
        'epochs': 600, 'patience': 150, 'batch': 16, 'imgsz': 800,
        'optimizer': 'AdamW', 'lr0': 0.001, 'mosaic': 0.0,
        'mixup': 0.15, 'copy_paste': copy_paste_used, 'cos_lr': True,
    },
}

with open(METRICS_JSON, 'w') as f:
    json.dump(dafe_metrics, f, indent=2)
print(f'DAFE metrics saved to: {METRICS_JSON}')

# Load YOLOv26n-only baseline
baseline_json = EVALS_DIR / 'yolov26n_neudet_results.json'
if baseline_json.exists():
    with open(baseline_json) as f:
        baseline = json.load(f)

    print(f'\n{"=" * 70}')
    print('COMPARISON: YOLOv26n + DAFE  vs  YOLOv26n (baseline)')
    print('=' * 70)

    b_map50 = baseline['map50']
    d_map50 = map50
    delta50 = d_map50 - b_map50
    sign50 = '+' if delta50 > 0 else ''
    print(f'\nmAP@0.5:     {b_map50*100:.1f}% -> {d_map50*100:.1f}%  ({sign50}{delta50*100:.1f}%)')

    b_map5095 = baseline['map50_95']
    d_map5095 = map50_95
    delta5095 = d_map5095 - b_map5095
    sign5095 = '+' if delta5095 > 0 else ''
    print(f'mAP@50:95:   {b_map5095*100:.1f}% -> {d_map5095*100:.1f}%  ({sign5095}{delta5095*100:.1f}%)')

    b_prec = baseline['precision']
    d_prec = precision
    d_prec_delta = d_prec - b_prec
    s_prec = '+' if d_prec_delta > 0 else ''
    print(f'Precision:   {b_prec:.4f} -> {d_prec:.4f}  ({s_prec}{d_prec_delta:.4f})')

    b_rec = baseline['recall']
    d_rec = recall
    d_rec_delta = d_rec - b_rec
    s_rec = '+' if d_rec_delta > 0 else ''
    print(f'Recall:      {b_rec:.4f} -> {d_rec:.4f}  ({s_rec}{d_rec_delta:.4f})')

    print(f'\nPer-class AP@0.5 delta:')
    print('-' * 55)
    print(f'  {"Class":<20} {"YOLOv26n":>10} {"+DAFE":>10} {"Delta":>10}')
    print('-' * 55)
    for cls_name, ap in per_class_ap50.items():
        b_ap = baseline['per_class_ap50'].get(cls_name, 0)
        delta = ap - b_ap
        ds = '+' if delta > 0 else ''
        print(f'  {cls_name:<20} {b_ap*100:>9.1f}% {ap*100:>9.1f}% {ds}{delta*100:>8.1f}%')
    print('=' * 70)

    # Overall verdict
    if d_map50 > b_map50:
        print(f'\nDAFE IMPROVED mAP@0.5 by +{delta50*100:.1f}%')
    elif d_map50 == b_map50:
        print(f'\nDAFE matched baseline mAP@0.5')
    else:
        print(f'\nDAFE reduced mAP@0.5 by {delta50*100:.1f}%')
else:
    print(f'\nBaseline not found at {baseline_json}')
    print('Run yolov26_neudet.ipynb first for comparison.')

DAFE metrics saved to: D:\DigiSteel-Yolo\DigiSteel-YOLO\evals\yolov26n_dafe_neudet_results.json

COMPARISON: YOLOv26n + DAFE  vs  YOLOv26n (baseline)

mAP@0.5:     81.0% -> 75.3%  (-5.8%)
mAP@50:95:   43.1% -> 41.7%  (-1.5%)
Precision:   0.7734 -> 0.7247  (-0.0487)
Recall:      0.7622 -> 0.7018  (-0.0603)

Per-class AP@0.5 delta:
-------------------------------------------------------
  Class                  YOLOv26n      +DAFE      Delta
-------------------------------------------------------
  crazing                   55.1%      35.8%    -19.4%
  inclusion                 84.7%      81.7%     -3.0%
  patches                   88.3%      83.6%     -4.8%
  pitted_surface            81.3%      76.8%     -4.5%
  rolled-in_scale           78.2%      76.3%     -1.8%
  scratches                 98.6%      97.6%     -1.1%

DAFE reduced mAP@0.5 by -5.8%


In [36]:
# Cell 8: Training Curves Visualization
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

if not RESULTS_CSV.exists():
    alt = RUNS_DIR / RUN_NAME / 'results.csv'
    if alt.exists():
        RESULTS_CSV = alt
    else:
        raise FileNotFoundError('results.csv not found')

df = pd.read_csv(RESULTS_CSV)
df.columns = df.columns.str.strip()
print(f'Loaded results: {len(df)} epochs')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('YOLOv26n + DAFE on NEU-DET — Training Curves', fontsize=14, fontweight='bold')

# Loss
loss_cols = [c for c in df.columns if 'loss' in c.lower()]
if loss_cols:
    ax = axes[0, 0]
    for col in loss_cols:
        ax.plot(df[col], label=col, alpha=0.8)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# mAP
map50_col = [c for c in df.columns if 'map50' in c.lower() and 'map50-' not in c.lower()]
map_col = [c for c in df.columns if 'map50-95' in c.lower()]
ax = axes[0, 1]
if map50_col:
    ax.plot(df[map50_col[0]], label='mAP@0.5', color='blue', linewidth=2)
if map_col:
    ax.plot(df[map_col[0]], label='mAP@0.5:0.95', color='red', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('mAP')
ax.set_title('mAP Progress')
ax.legend()
ax.grid(True, alpha=0.3)

# Learning Rate
lr_col = [c for c in df.columns if 'lr' in c.lower() or 'pg0' in c.lower()]
ax = axes[1, 0]
if lr_col:
    ax.plot(df[lr_col[0]], color='green', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedule')
    ax.grid(True, alpha=0.3)

# Precision & Recall
prec_col = [c for c in df.columns if 'precision' in c.lower()]
rec_col = [c for c in df.columns if 'recall' in c.lower()]
ax = axes[1, 1]
if prec_col:
    ax.plot(df[prec_col[0]], label='Precision', color='purple', linewidth=2)
if rec_col:
    ax.plot(df[rec_col[0]], label='Recall', color='orange', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Score')
ax.set_title('Precision and Recall')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RUNS_DIR / RUN_NAME / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Curves saved to: {RUNS_DIR / RUN_NAME / "training_curves.png"}')

Loaded results: 555 epochs


<Figure size 1400x1000 with 4 Axes>

Curves saved to: D:\DigiSteel-Yolo\DigiSteel-YOLO\runs\detect\yolov26n_dafe_neudet\training_curves.png


In [37]:
# Cell 9: 3-Way Per-Class Comparison Bar Chart
import matplotlib.pyplot as plt
import numpy as np
import json

# Load DAFE results from saved file (survives kernel restart)
with open(METRICS_JSON) as f:
    dafe_results = json.load(f)

yolo26_json = EVALS_DIR / 'yolov26n_neudet_results.json'
baseline_json = EVALS_DIR / 'fresh_baseline_results.json'

has_yolo26 = yolo26_json.exists()
has_baseline = baseline_json.exists()

if has_yolo26:
    with open(yolo26_json) as f:
        yolo26_results = json.load(f)

if has_baseline:
    with open(baseline_json) as f:
        baseline_results = json.load(f)

classes = list(dafe_results['per_class_ap50'].keys())
dafe_aps = [dafe_results['per_class_ap50'][c] * 100 for c in classes]

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(classes))

# Calculate bar positions dynamically based on available models
models_data = []
if has_baseline:
    baseline_aps = [baseline_results['per_class_ap50'][c] * 100 for c in classes]
    models_data.append((baseline_aps, 'YOLOv11n (baseline)', '#4ECDC4'))
if has_yolo26:
    yolo26_aps = [yolo26_results['per_class_ap50'][c] * 100 for c in classes]
    models_data.append((yolo26_aps, 'YOLOv26n', '#FF6B6B'))
models_data.append((dafe_aps, 'YOLOv26n + DAFE', '#FFD93D'))

n_models = len(models_data)
width = min(0.25, 0.7 / max(n_models, 1))  # adaptive width to prevent overlap

for i, (aps, label, color) in enumerate(models_data):
    offset = (i - (n_models - 1) / 2) * width
    bars = ax.bar(x + offset, aps, width, label=label, color=color,
                  edgecolor='black', linewidth=0.5)
    # Add value labels
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., h + 1, f'{h:.1f}',
                ha='center', va='bottom', fontsize=7)

ax.set_ylabel('AP@0.5 (%)', fontsize=12)
ax.set_title('Per-Class AP@0.5: YOLOv26n + DAFE vs Baselines on NEU-DET', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig(RUNS_DIR / RUN_NAME / 'per_class_3way_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved to: {RUNS_DIR / RUN_NAME / "per_class_3way_comparison.png"}')

<Figure size 1400x700 with 1 Axes>

Chart saved to: D:\DigiSteel-Yolo\DigiSteel-YOLO\runs\detect\yolov26n_dafe_neudet\per_class_3way_comparison.png


In [38]:
# Cell 10: Summary Table
import json

# Load DAFE results from saved file (survives kernel restart)
with open(METRICS_JSON) as f:
    dafe_data = json.load(f)

map50 = dafe_data['map50']
map50_95 = dafe_data['map50_95']
per_class_ap50 = dafe_data['per_class_ap50']
training_time = dafe_data.get('training_time_hours', 0) * 3600  # convert back to seconds

print('=' * 70)
print('EXPERIMENT SUMMARY')
print('=' * 70)
print(f'{"Model":<25} {"mAP@0.5":>10} {"mAP@50:95":>10} {"Params":>10}')
print('-' * 70)

# YOLOv11n baseline
bl_json = EVALS_DIR / 'fresh_baseline_results.json'
if bl_json.exists():
    with open(bl_json) as f:
        bl = json.load(f)
    print(f'{"YOLOv11n (baseline)":<25} {bl["map50"]*100:>9.1f}% {bl["map50_95"]*100:>9.1f}% {"2.6M":>10}')

# YOLOv26n
y26_json = EVALS_DIR / 'yolov26n_neudet_results.json'
y26_data = None
if y26_json.exists():
    with open(y26_json) as f:
        y26_data = json.load(f)
    print(f'{"YOLOv26n":<25} {y26_data["map50"]*100:>9.1f}% {y26_data["map50_95"]*100:>9.1f}% {"2.4M":>10}')

# YOLOv26n + DAFE (this experiment) — use actual measured params
dafe_params = 2_629_790  # from Cell 3 verification
print(f'{"YOLOv26n + DAFE":<25} {map50*100:>9.1f}% {map50_95*100:>9.1f}% {dafe_params/1e6:.1f}M')

print('-' * 70)

# Delta from YOLOv26n baseline
if y26_data is not None:
    delta = map50 - y26_data['map50']
    sign = '+' if delta > 0 else ''
    print(f'\nDAFE impact on YOLOv26n: {sign}{delta*100:.1f}% mAP@0.5')

    # Per-class breakdown
    print(f'\nPer-class impact:')
    for cls_name, ap in per_class_ap50.items():
        b_ap = y26_data['per_class_ap50'].get(cls_name, 0)
        d = ap - b_ap
        s = '+' if d > 0 else ''
        improved = '+' if d > 0 else ('-' if d < 0 else '=')
        print(f'  {improved} {cls_name:<20} {s}{d*100:.1f}%')

print(f'\nTraining time: {training_time/3600:.1f} hours')
print('=' * 70)

EXPERIMENT SUMMARY
Model                        mAP@0.5  mAP@50:95     Params
----------------------------------------------------------------------
YOLOv11n (baseline)            78.8%      45.2%       2.6M
YOLOv26n                       81.0%      43.1%       2.4M
YOLOv26n + DAFE                75.3%      41.7% 2.6M
----------------------------------------------------------------------

DAFE impact on YOLOv26n: -5.8% mAP@0.5

Per-class impact:
  - crazing              -19.4%
  - inclusion            -3.0%
  - patches              -4.8%
  - pitted_surface       -4.5%
  - rolled-in_scale      -1.8%
  - scratches            -1.1%

Training time: 5.3 hours
